In [1]:
# 1. imports
import asyncio, json, sys, os
sys.path.insert(0, os.path.abspath(".."))
import aiohttp
from lib.execution import ExecutionClient

KEY_FILE = "../keys/truemarkets-api-key-edd1691b.json"
BASE_URL = "https://api.truemarkets.co"

In [2]:
# 2. auth
bot = ExecutionClient(key_file=KEY_FILE, base_url=BASE_URL)
session = aiohttp.ClientSession()
await bot.authenticate(session)
print("authenticated, access_token len:", len(bot._access_token or ""))

authenticated, access_token len: 862


In [3]:
# 3. portfolio (mirrors portfolio.py, using the corrected response schemas)
balances = await bot._get(session, "/v1/conductor/balances")
print("=== /balances (CeFi) ===")
for b in (balances or {}).get("balances", []):
    print(f"  {b.get('asset_name') or b.get('asset_id'):8} {b.get('balance')}")

unified = await bot._get(session, "/v1/conductor/balances/unified")
print("\n=== /balances/unified (CeFi + DeFi) ===")
for b in (unified or {}).get("data", []):
    chain = b.get("chain") or "cefi"
    print(f"  {b['symbol']:8} chain={chain:8} available={b['available']:>14} "
          f"held={b['held']:>10} tradeable={b['tradeable']}")

=== /balances (CeFi) ===

=== /balances/unified (CeFi + DeFi) ===
  PYUSD    chain=cefi     available= 648.329672066 held=2.6454041172 tradeable=False
  BTC      chain=cefi     available=    0.00137842 held=         0 tradeable=True
  PYUSD    chain=solana   available=      1.838928 held=         0 tradeable=True


In [4]:
# 4. list assets (catalog, per venue). DeFi is large and paginated -
# this only pulls page 1; raise limit or walk pagination/next_cursor for more.
for venue in ["cefi"]:#, "defi"):
    resp = await bot._get(session, f"/v1/conductor/assets?venue={venue}")
    print(resp)
    items = (resp or {}).get("data", [])
    print(f"=== {venue.upper()} assets (showing {len(items)}) ===")
    for a in items[:25]:
        chain = f" chain={a['chain']}" if a.get("chain") else ""
        print(f"  {a['symbol']:8} tradeable={str(a['tradeable']):5}{chain}")
    if len(items) > 25:
        print(f"  ... +{len(items) - 25} more")

{'data': [{'id': 'b2538174-aae3-4b55-b79f-ac177c99f816', 'chain': None, 'address': None, 'symbol': 'BTC', 'name': 'Bitcoin', 'decimals': 8, 'slug': 'bitcoin', 'is_active': True, 'status': 'active', 'description': 'Bitcoin is the world\'s first decentralized cryptocurrency, created in 2009 by the pseudonymous Satoshi Nakamoto. It enables peer-to-peer electronic cash transactions without intermediaries like banks or governments, operating on a blockchain secured by Proof of Work mining and the SHA-256 cryptographic algorithm. \r\n\r\nWith a fixed supply cap of 21 million coins and programmatic halvings every four years that reduce miner rewards, Bitcoin is designed as a deflationary digital asset often called "digital gold." Its value stems from solving the double-spending problem without trusted intermediaries, creating the first truly scarce digital asset with censorship resistance and permissionless access that no government, corporation, or individual can control.\r\n\r\nBitcoin oper

In [ ]:
# 5. test trade (mirrors test.py: market buy $1 of BTC via USDC)
# USDC isn't a CeFi asset on this account at all (catalog only has BTC/ETH/PYUSD,
# see cell 4) and CeFi cash balance is held in PYUSD, not USDC — quote in PYUSD.
order = await bot.place_order(
    session, base_asset="ETH", quote_asset="PYUSD",
    side="buy", qty=".005", qty_unit="base", order_type="market",
)
print("order result:", order)
if order and order.get("order_id"):
    status = await bot.get_order_status(session, order["order_id"])
    print("status:", status)

In [ ]:
# 6. test trade [limit trade](mirrors test.py: market buy $1 of BTC via USDC)
# USDC isn't a CeFi asset on this account at all (catalog only has BTC/ETH/PYUSD,
# see cell 4) and CeFi cash balance is held in PYUSD, not USDC — quote in PYUSD.
# Limit orders are CeFi-only (no DeFi funding-bridge fallback), so quoting in a
# nonexistent asset 503'd outright instead of routing around it like the market
# buy above sometimes does.
order = await bot.place_order(
    session, base_asset="ETH", quote_asset="PYUSD",
    side="buy", qty=".0001", qty_unit="base", order_type="limit", price = "2500"
)
print("order result:", order)
if order and order.get("order_id"):
    status = await bot.get_order_status(session, order["order_id"])
    print("status:", status)

In [6]:
import time
# 6. test trade [limit trade](mirrors test.py: market buy $1 of BTC via USDC)
# Known from live testing: this account has no CeFi USDC balance, so this
# will very likely come back status="failed" with no funds moved, until
# USDC is funded directly into CeFi custody (see prior investigation).
order = await bot.place_order(
    session, base_asset="BTC", quote_asset="USDC",
    side="sell", qty=".0001", qty_unit="base", order_type="limit", price = "62550"
)
#print("order result:", order)

if order and order.get("order_id"):
    time.sleep(3)
    status = await bot.get_order_status(session, order["order_id"])
    print("status:", status)

bot.cancel_order(session, order.get("order_id"))

time.sleep(3)

order = await bot.place_order(
    session, base_asset="BTC", quote_asset="USDC",
    side="sell", qty=".0001", qty_unit="base", order_type="limit", price = "62550", venue = "defi"
)
#print("order result:", order)

if order and order.get("order_id"):
    time.sleep(3)
    status = await bot.get_order_status(session, order["order_id"])
    print("status:", status)

bot.cancel_order(session, order.get("order_id"))

{'order_id': '2b8d7058-22a2-4b18-bdb9-071f9db9495f', 'status': 'pending'}
status: active


/var/folders/22/t8xqqxmj2g5c4pnhgzh4g8vw0000gq/T/ipykernel_55831/2279520740.py:17: RuntimeWarning: coroutine 'ExecutionClient.cancel_order' was never awaited
  bot.cancel_order(session, order.get("order_id"))


{'order_id': 'c13f2f9c-d37e-4949-8969-8df5467638a1', 'status': 'pending'}
status: active


<coroutine object ExecutionClient.cancel_order at 0x1138168a0>

In [8]:
print(order)

{'order_id': 'eae3c3d9-f18c-4a10-b13a-0dc2c14c2b82', 'status': 'pending'}


In [4]:
status = await bot.get_order(session, "39f39d12-3131-46f0-8468-62773446779c")
print("status:", status)

status: {'order_id': '39f39d12-3131-46f0-8468-62773446779c', 'base_asset': 'BTC', 'quote_asset': 'PYUSD', 'asset_id': 'b2538174-aae3-4b55-b79f-ac177c99f816', 'asset_symbol': 'BTC', 'type': 'limit', 'side': 'sell', 'price': '63199', 'qty': '0.005', 'executed_qty': '0.005', 'leaves_qty': '0', 'executed_vwap': '64200', 'fee': '0.1284', 'venue': 'cefi', 'status': 'complete', 'created_at': '2026-07-11T06:37:58.193418Z'}


In [4]:
status = await bot.get_order(session, "3730015d-6fc2-48a6-9cd8-1d6c4a55b56b")
print("status:", status)

status: {'order_id': '3730015d-6fc2-48a6-9cd8-1d6c4a55b56b', 'base_asset': 'BTC', 'quote_asset': 'USDC', 'type': 'limit', 'side': 'sell', 'price': '58589.6', 'qty': '0.0001', 'executed_qty': '0.00007', 'leaves_qty': '0', 'executed_vwap': '58673', 'fee': '0.001642844', 'venue': 'cefi', 'status': 'canceled', 'created_at': '2026-06-26T13:20:19.497455Z'}


In [4]:
data = await bot._get(session, "/v1/account/deposits/addresses")
print(data)


{'data': [{'id': 'c0e4b732-750d-4501-b9fe-58f402b26393', 'network': 'bitcoin', 'address': 'bc1qqdles53q5ljq5cf5hzun83q8wlsdl5s6v90czw', 'compatible_networks': [], 'created_at': '2026-06-13T02:46:45.562277Z'}, {'id': '19ad5e92-8e32-4b4d-a3e1-6e2621f3cff6', 'network': 'ethereum', 'address': '0xb151224F746DE86364D25A726C826944d590c142', 'compatible_networks': ['ink', 'avalanche', 'base', 'xlayer', 'polygon_pos', 'arbitrum_one'], 'created_at': '2026-06-13T02:46:45.871548Z'}, {'id': '4e288bc5-735b-43c9-bdee-31575bdabe75', 'network': 'solana', 'address': '8nBwV7jDhtBU9kupwksTcYf7LwEFaVW6U452shWmSZUr', 'compatible_networks': [], 'created_at': '2026-06-13T02:46:46.206817Z'}], 'pagination': {'next_cursor': None, 'limit': 3}}
